In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import heapq
import itertools
import time
from abc import ABC, abstractmethod
from collections import deque

# ==========================================
# 1. CORE ENGINE (AdvancedOrderBook)
# ==========================================

class AdvancedOrderBook:
    def __init__(self):
        self.bids = []  
        self.asks = []  
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side,
                'aggressor': 'market' if limit_price == float('inf') else 'limit'
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty

# ==========================================
# 2. ANALYTICS ENGINE (Updated for Spread)
# ==========================================

class AnalyticsEngine:
    def __init__(self):
        self.tape = []      
        self.snapshots = [] # To record spread stability

    def log_trade(self, timestamp, price, qty):
        self.tape.append({
            'timestamp': timestamp,
            'price': price,
            'qty': qty
        })

    def log_snapshot(self, timestamp, best_bid, best_ask):
        spread = (best_ask - best_bid) if (best_bid and best_ask) else np.nan
        self.snapshots.append({
            'timestamp': timestamp,
            'spread': spread
        })

    def get_tape_dataframe(self):
        df = pd.DataFrame(self.tape)
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df.set_index('timestamp', inplace=True)
        return df

    def get_snapshot_dataframe(self):
        df = pd.DataFrame(self.snapshots)
        if not df.empty:
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df.set_index('timestamp', inplace=True)
        return df

# ==========================================
# 3. AGENT ZOO (Noise + Momentum + MarketMaker)
# ==========================================

class Agent(ABC):
    def __init__(self, agent_id, cash, inventory):
        self.agent_id = agent_id
        self.cash = cash
        self.inventory = inventory
        
    @abstractmethod
    def get_action(self, market_snapshot, fair_value=None):
        pass

class NoiseTrader(Agent):
    """ Day 7: Liquidity Consumer """
    def __init__(self, agent_id, cash, inventory, arrival_rate=0.3, volatility=0.02):
        super().__init__(agent_id, cash, inventory)
        self.arrival_rate = arrival_rate
        self.volatility = volatility 

    def get_action(self, market_snapshot, fair_value):
        if np.random.random() > self.arrival_rate: return None
        side = 'buy' if np.random.random() > 0.5 else 'sell'
        noise = np.random.normal(0, self.volatility * fair_value)
        order_price = round(fair_value + noise, 2)
        qty = np.random.randint(1, 5)
        return {'agent_id': self.agent_id, 'side': side, 'price': order_price, 'qty': qty, 'type': 'limit'}

class MomentumTrader(Agent):
    """ Day 8: Trend Follower """
    def __init__(self, agent_id, cash, inventory, lookback=20, threshold=0.5):
        super().__init__(agent_id, cash, inventory)
        self.lookback = lookback
        self.threshold = threshold
        self.price_history = deque(maxlen=lookback)

    def get_action(self, market_snapshot, fair_value=None):
        mid_price = market_snapshot.get('mid_price')
        if mid_price is None: return None
        self.price_history.append(mid_price)
        if len(self.price_history) < self.lookback: return None

        sma = sum(self.price_history) / len(self.price_history)
        qty = 10 
        
        if mid_price > sma + self.threshold:
            return {'agent_id': self.agent_id, 'side': 'buy', 'price': round(mid_price + 0.5, 2), 'qty': qty, 'type': 'limit'}
        elif mid_price < sma - self.threshold:
            return {'agent_id': self.agent_id, 'side': 'sell', 'price': round(mid_price - 0.5, 2), 'qty': qty, 'type': 'limit'}
        return None

class MarketMakerAgent(Agent):
    """
    Day 9: Liquidity Provider.
    Strategy: Avellaneda-Stoikov (Simplified).
    - Quotes Bid and Ask around the Mid Price.
    - Adjusts quotes based on Inventory (Skewing).
    - If Long Inventory -> Lower quotes (Sell faster, Buy slower).
    - If Short Inventory -> Raise quotes (Buy faster, Sell slower).
    """
    def __init__(self, agent_id, cash, inventory, half_spread=0.5, skew_factor=0.05):
        super().__init__(agent_id, cash, inventory)
        self.half_spread = half_spread
        self.skew_factor = skew_factor # How much to shift price per unit of inventory

    def get_action(self, market_snapshot, fair_value=None):
        mid_price = market_snapshot.get('mid_price')
        if mid_price is None: return None # Can't quote if no price

        # Inventory Risk Adjustment
        # If Inventory > 0 (Long), skew is negative -> prices drop -> easier to sell, harder to buy
        skew = -1 * self.inventory * self.skew_factor
        
        # Calculate Quotes
        bid_price = round(mid_price - self.half_spread + skew, 2)
        ask_price = round(mid_price + self.half_spread + skew, 2)
        
        # Avoid Crossed Quotes (Bid >= Ask)
        if bid_price >= ask_price:
            ask_price = bid_price + 0.01

        # Return a LIST of two orders (Bid and Ask)
        return [
            {'agent_id': self.agent_id, 'side': 'buy', 'price': bid_price, 'qty': 5, 'type': 'limit'},
            {'agent_id': self.agent_id, 'side': 'sell', 'price': ask_price, 'qty': 5, 'type': 'limit'}
        ]

# ==========================================
# 4. SIMULATION LOOP (Scenario: Stability Test)
# ==========================================

SEED = 42
NUM_STEPS = 2000
OUTPUT_FILENAME = "day9_market_maker_report.pdf"

def generate_fair_value_series(steps, start_price=100.0, drift=0.0, vol=0.0005):
    prices = [start_price]
    for _ in range(steps):
        change = np.random.normal(drift, vol)
        prices.append(prices[-1] * (1 + change))
    return prices

def process_data(df_tape):
    if df_tape.empty: return pd.DataFrame()
    ohlc = df_tape['price'].resample('1min').ohlc()
    ohlc['volume'] = df_tape['qty'].resample('1min').sum()
    ohlc.dropna(inplace=True)
    return ohlc

def generate_report(ohlc_data, df_snapshots, fair_values):
    print(f"Generating report: {OUTPUT_FILENAME}...")
    with PdfPages(OUTPUT_FILENAME) as pdf:
        # Page 1: Price Action
        fig, ax = plt.subplots(figsize=(10, 6))
        times = pd.date_range(start="2025-01-01 09:30", periods=len(fair_values), freq="S")
        ax.plot(times, fair_values, color='orange', alpha=0.3, label='Fair Value')

        up = ohlc_data[ohlc_data.close >= ohlc_data.open]
        down = ohlc_data[ohlc_data.close < ohlc_data.open]
        ax.bar(up.index, up.close - up.open, bottom=up.open, width=0.0005, color='green')
        ax.vlines(up.index, up.low, up.high, color='green')
        ax.bar(down.index, down.close - down.open, bottom=down.open, width=0.0005, color='red')
        ax.vlines(down.index, down.low, down.high, color='red')
        
        ax.set_title("Day 9: Market Makers Stabilizing Prices")
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        pdf.savefig(fig)
        plt.close()
        
        # Page 2: Spread Stability
        fig2, ax2 = plt.subplots(figsize=(10, 6))
        # Resample spread to smooth it out for plotting
        spread_series = df_snapshots['spread'].resample('10s').mean()
        ax2.plot(spread_series.index, spread_series.values, color='purple', label='Bid-Ask Spread')
        ax2.set_title("Spread Stability (Should be tight)")
        ax2.set_ylabel("Spread Amount")
        ax2.legend()
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        pdf.savefig(fig2)
        plt.close()

def run_simulation():
    print(f"--- Starting Day 9 Simulation (Market Makers) ---")
    random.seed(SEED)
    np.random.seed(SEED)
    
    engine = AdvancedOrderBook()
    analytics = AnalyticsEngine()
    
    fair_values = generate_fair_value_series(NUM_STEPS, start_price=100.0)
    
    # AGENT SETUP:
    # 20 Noise Traders (Consumers)
    # 2 Market Makers (Providers) - They fight for the spread!
    agents = [NoiseTrader(i, 100000, 1000) for i in range(20)]
    agents += [MarketMakerAgent(100+i, 500000, 0, half_spread=0.2) for i in range(2)]
    
    current_time = datetime(2025, 1, 1, 9, 30)
    
    for i in range(NUM_STEPS):
        current_time += timedelta(seconds=1)
        current_fair_value = fair_values[i]
        
        # 1. Market Snapshot
        best_bid = -engine.bids[0][0] if engine.bids else None
        best_ask = engine.asks[0][0] if engine.asks else None
        mid_price = (best_bid + best_ask) / 2 if (best_bid and best_ask) else current_fair_value
        
        # Log Snapshot for Spread Analysis
        analytics.log_snapshot(current_time, best_bid, best_ask)
        
        snapshot = {'best_bid': best_bid, 'best_ask': best_ask, 'mid_price': mid_price}
        
        # 2. Agents Act
        random.shuffle(agents)
        for agent in agents:
            actions = agent.get_action(snapshot, current_fair_value)
            
            # Handle Market Makers returning a LIST of orders
            if isinstance(actions, list):
                for order in actions:
                    engine.submit_order(order['side'], order['qty'], order['price'], order['type'])
                    
                    # Hack: Market Makers update internal inventory immediately for simulation
                    # In real engine, they would wait for execution confirmation
                    # Here we assume partial fills or just tracking intent for skew logic
                    if order['side'] == 'buy': agent.inventory += 1 # Simplified assumption
                    else: agent.inventory -= 1
            elif actions:
                engine.submit_order(actions['side'], actions['qty'], actions['price'], actions['type'])

        # 3. Log Trades
        while engine.trades:
            trade = engine.trades.pop(0)
            analytics.log_trade(current_time, trade['price'], trade['qty'])

    print("Simulation complete.")
    return analytics.get_tape_dataframe(), analytics.get_snapshot_dataframe(), fair_values

if __name__ == "__main__":
    df_tape, df_snapshots, fv_data = run_simulation()
    if not df_tape.empty:
        df_ohlc = process_data(df_tape)
        if not df_ohlc.empty:
            generate_report(df_ohlc, df_snapshots, fv_data)
            print(f"Done. Check {OUTPUT_FILENAME}")
    else:
        print("No trades.")

--- Starting Day 9 Simulation (Market Makers) ---
Simulation complete.
Generating report: day9_market_maker_report.pdf...
Done. Check day9_market_maker_report.pdf


C:\Users\Asus User\AppData\Local\Temp\ipykernel_48024\3001002281.py:228: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  times = pd.date_range(start="2025-01-01 09:30", periods=len(fair_values), freq="S")
